# Gem Shuffle: Attention Experiment Analysis

> **SIMULATED – replace by playing the game.** If `data/rounds.csv` and `data/games.csv` do not exist yet, this notebook makes **fake demo data** (in the "Making the SIMULATED demo data" section) so that every cell still runs. The numbers in the demo are random, so none of the findings mean anything until you play the game and the real files are created.

Each heading explains the cell below it. Run the cells from the top, one after another.

## Setup
We import the libraries we need, and the cleaning functions from `data.py`. This notebook never writes files, so running it is safe.

In [ ]:
import os
import numpy as np
import pandas as pd
import data

ROUNDS_PATH = os.path.join("data", "rounds.csv")
GAMES_PATH = os.path.join("data", "games.csv")

## Making the SIMULATED demo data
This function plays fake games, using the cleaned levels and gem names. A game ends when the player runs out of lives or passes all levels. We use `np.random` for the random choices, and `np.random.seed(1)` so the fake numbers are the same every time.

**SIMULATED – replace by playing the game.**

In [ ]:
def make_simulated_data(number_of_games=40):
    # SIMULATED – replace by playing the game
    levels = data.load_levels().to_dict("records")
    gem_names = list(data.load_gems()["gem"])
    np.random.seed(1)

    round_rows = []
    game_rows = []
    for game_id in range(1, number_of_games + 1):
        lives = 3
        level_index = 0
        score = 0
        right_flags = []
        gems_found = set()
        game_start = len(round_rows)
        while lives > 0 and level_index < len(levels):
            level = levels[level_index]
            cups = int(level["cups"])
            # Harder levels (more fake swaps) are a bit more likely to be wrong
            chance_right = 0.9 - 0.05 * int(level["fakes"])
            correct = np.random.rand() < chance_right
            answer = np.random.randint(0, cups)
            if correct:
                chosen = answer
            else:
                chosen = (answer + np.random.randint(1, cups)) % cups
            gem = gem_names[np.random.randint(0, len(gem_names))]
            reaction = round(np.random.uniform(0.8, 2.5) + (0 if correct else 0.4), 2)
            points = 20 * int(level["level"]) if correct else 0
            score += points
            right_flags.append(bool(correct))
            if correct:
                gems_found.add(gem)
                level_index += 1
            else:
                lives -= 1
            round_rows.append({
                "game_id": game_id, "player": "Demo{}".format(game_id), "date": "SIMULATED",
                "level": int(level["level"]), "cups": cups, "swaps": int(level["swaps"]),
                "fakes": int(level["fakes"]), "swap_ms": int(level["swap_ms"]), "gem": gem,
                "start_slot": np.random.randint(0, cups), "answer_slot": answer,
                "chosen_slot": chosen, "correct": int(correct), "reaction_s": reaction,
                "points": points})

        # Longest run of correct answers in a row
        best_streak = 0
        run = 0
        for flag in right_flags:
            run = run + 1 if flag else 0
            best_streak = max(best_streak, run)

        # Average reaction time for this game's rounds only
        reactions = [r["reaction_s"] for r in round_rows[game_start:]]
        game_rows.append({
            "game_id": game_id, "player": "Demo{}".format(game_id), "date": "SIMULATED",
            "score": score, "level_reached": level_index + 1 if lives == 0 else level_index,
            "rounds": len(right_flags), "correct": sum(right_flags),
            "best_streak": best_streak,
            "avg_reaction_s": round(float(np.mean(reactions)), 2),
            "gems_won": ";".join(sorted(gems_found)),
            "result": "lost" if lives == 0 else "won"})

    return pd.DataFrame(round_rows), pd.DataFrame(game_rows)

## 1. Load and inspect
We use the real files if they exist. If not, we use the **SIMULATED** demo data. Then we look at the shape, the column types, and the first and last 4 rows.

In [ ]:
if os.path.exists(ROUNDS_PATH) and os.path.exists(GAMES_PATH):
    rounds = pd.read_csv(ROUNDS_PATH)
    games = pd.read_csv(GAMES_PATH)
    print("Using the REAL data from the game.")
else:
    rounds, games = make_simulated_data()
    print("SIMULATED – replace by playing the game. No real files were found.")

print("rounds shape:", rounds.shape)
print("games shape:", games.shape)
print()
print(rounds.dtypes)
print()
print(rounds.head(4))
print(rounds.tail(4))

### 1b. The games table
The same checks for the `games` table, one row per game.

In [ ]:
print("games shape:", games.shape)
print(games.dtypes)
print(games.head(4))
print(games.tail(4))

## 2. Cleaning demo
We show the messy raw levels and gems files before cleaning. The `??` and `##` symbols are first turned into NaN (missing), so that `isnull()` can count them. The full cleaning is done by `data.py`.

In [ ]:
raw_levels = pd.read_csv(os.path.join("data", "levels_raw.csv"))
raw_gems = pd.read_csv(os.path.join("data", "gems_raw.csv"))

print("Levels: missing values BEFORE cleaning")
print(raw_levels.replace(["??", "##"], np.nan).isnull().sum())
print()
print("Gems: missing values BEFORE cleaning")
print(raw_gems.replace(["??", "##"], np.nan).isnull().sum())

### 2b. The clean tables
`load_levels()` and `load_gems()` return the cleaned tables. `load_levels()` also prints the missing counts before and after it fills the gaps.

In [ ]:
clean_levels = data.load_levels()
clean_gems = data.load_gems()

print("Clean levels (10 rows):")
print(clean_levels)
print()
print("Clean gems (5 rows):")
print(clean_gems)

## 3. Descriptive statistics
We describe the reaction times (`reaction_s`) and the scores (`score`). The mean is the average. The median is the middle value. The mode is the most common value. The variance shows how spread out the values are. The skew shows if the values lean to one side.

In [ ]:
print("Summary of reaction_s, from rounds")
print(rounds["reaction_s"].describe())
print()
print("Summary of score, from games")
print(games["score"].describe())
print()

for name, column in [("reaction_s", rounds["reaction_s"]), ("score", games["score"])]:
    print("{}: mean={}, median={}, mode={}, variance={}, skew={}".format(
        name, round(column.mean(), 2), column.median(), column.mode()[0],
        round(column.var(), 2), round(column.skew(), 2)))

## 4. Experiment questions
These are the questions the experiment is about. Each cell answers one of them with a table.

### 4.1 Accuracy by level, by number of bowls and by fake swaps
Accuracy is the percentage of correct rounds (`correct` is 1 for a right answer, 0 for a wrong one). `groupby` puts the rounds into groups, and `mean()` gives the share of correct answers in each group.

In [ ]:
print("Accuracy % by level")
print((rounds.groupby("level")["correct"].mean() * 100).round(1))
print()
print("Accuracy % by number of bowls")
print((rounds.groupby("cups")["correct"].mean() * 100).round(1))
print()
print("Accuracy % by number of fake swaps")
print((rounds.groupby("fakes")["correct"].mean() * 100).round(1))

### 4.2 Position bias in 3-bowl rounds
If the player has no bias, every slot should be chosen about one third of the time. We count how often each slot was chosen in the 3-bowl rounds, and compare that with the fair share of 1/3.

In [ ]:
three = rounds[rounds["cups"] == 3]
chosen_counts = three.groupby("chosen_slot").size()
fair_share = len(three) / 3

print("Times each slot was chosen (3-bowl rounds):")
print(chosen_counts)
print("Fair share for each slot:", round(fair_share, 1))

### 4.3 Shuffle fairness
We count where the gem really ended up (`answer_slot`) across all rounds. If the shuffle is fair, no slot should be much more common than the others.

In [ ]:
print("Times the gem ended in each slot (all rounds):")
print(rounds.groupby("answer_slot").size())

### 4.4 Reaction time: correct and wrong rounds
We compare the average reaction time of wrong rounds (`correct` = 0) and correct rounds (`correct` = 1).

In [ ]:
print("Average reaction time (seconds) by correct (0 = wrong, 1 = right)")
print(rounds.groupby("correct")["reaction_s"].mean().round(2))

### 4.5 Did the gem start where it ended?
`start_slot == answer_slot` is True when the gem is in the same slot at the end as it was at the start. Taking the mean of a True/False array gives the share of rounds that are True.

In [ ]:
same_slot = rounds["start_slot"] == rounds["answer_slot"]
print("Share of rounds where the gem ended in its starting slot: {}%".format(round(same_slot.mean() * 100, 1)))
print("Fair share for 3 bowls would be about 33%")

## 5. Extremes
`idxmin()` gives the row number of the smallest value, and `idxmax()` gives the row number of the largest. We use them to find the fastest correct round and the slowest round. Then `sort_values` makes the leaderboard.

In [ ]:
correct_rounds = rounds[rounds["correct"] == 1]
fastest_row = correct_rounds["reaction_s"].idxmin()
slowest_row = rounds["reaction_s"].idxmax()

print("Fastest correct round:")
print(rounds.loc[fastest_row, ["player", "level", "reaction_s"]])
print()
print("Slowest round (right or wrong):")
print(rounds.loc[slowest_row, ["player", "level", "reaction_s", "correct"]])
print()
print("Leaderboard (top 10 by score):")
leaderboard = games.sort_values("score", ascending=False).reset_index(drop=True)
print(leaderboard[["player", "score", "level_reached", "result"]].head(10))

## 6. Correlation
`corr()` gives a table of correlations. A value near +1 means the two columns rise together, near -1 means one falls as the other rises, and near 0 means no straight-line link.

In [ ]:
columns = ["level", "cups", "swaps", "fakes", "swap_ms", "reaction_s", "correct"]
print(rounds[columns].corr().round(2))

## 7. Pearson r by hand
Pearson r measures the straight-line link between two columns. Here we calculate it with the formula, using NumPy arrays, for `swaps` (x) and `correct` (y):

r = (n·Σxy − Σx·Σy) / √((n·Σx² − (Σx)²)(n·Σy² − (Σy)²))

Then we compare it with `.corr()` from pandas. The two numbers should match.

In [ ]:
x = np.array(rounds["swaps"], dtype=float)
y = np.array(rounds["correct"], dtype=float)
n = len(x)

top = n * np.sum(x * y) - np.sum(x) * np.sum(y)
bottom = np.sqrt((n * np.sum(x ** 2) - np.sum(x) ** 2) * (n * np.sum(y ** 2) - np.sum(y) ** 2))
r_by_hand = top / bottom

print("Pearson r by hand:", round(r_by_hand, 4))
print("Pearson r with .corr():", round(rounds["swaps"].corr(rounds["correct"]), 4))

## 8. New column: shuffle length
We make a new column, `shuffle_s`, which is the time the shuffle takes in seconds: (swaps + fakes) × swap_ms ÷ 1000. Then we look at accuracy by shuffle length, and the top 5 games by score.

In [ ]:
rounds["shuffle_s"] = (rounds["swaps"] + rounds["fakes"]) * rounds["swap_ms"] / 1000

print("Accuracy % by shuffle length (seconds)")
print((rounds.groupby("shuffle_s")["correct"].mean() * 100).round(1))
print()
print("Top 5 games by score")
print(games.sort_values("score", ascending=False)[["player", "score", "level_reached", "result"]].head(5))

## 9. Findings
Fill in the blanks after you have played the game and run the notebook on the **real** data. Use the tables above to support each answer. Do not write findings from the SIMULATED demo data.

1. Does accuracy fall as levels get harder? Answer: ____________________
2. Do players show a bias towards one bowl position in 3-bowl rounds? Answer: ____________________
3. Are correct answers faster or slower than wrong answers? By how much? Answer: ____________________
4. Is there a link between the number of swaps and accuracy (Pearson r)? Answer: ____________________
5. Does a longer shuffle make accuracy worse? Answer: ____________________
6. One thing that surprised us: ____________________